# Time optimization of the app development process

IS6003 final assignment. Research method: DSRM (Peffers et al., 2007). Process analysis follows *Fundamentals of BPM*, 2nd ed. (Dumas et al.). Page numbers are the book's printed page numbers.

This notebook runs the whole study from start to end with one seed:

1. check the models against the book's modeling rules;
2. simulate the corrected as-is model and write the event log;
3. measure the baseline and cross-check it with the book's flow analysis;
4. load the value-added classification;
5. train the ML models and rank the drivers of delay and rework;
6. compare those drivers with the effects we planted in the simulator;
7. load the redesign: the change catalog and the to-be models;
8. simulate the as-is process and every combination of changes, and read off the saving and the ablation;
9. write `web/results.json` for the page.

All logic lives in the `bpmopt` package and is tested there. The cells below only call it and show the output. A full run takes about 8 minutes, most of it in step 8.

In [1]:
from pathlib import Path

import pandas as pd

from bpmopt.analysis import baseline_metrics, flow_analysis_check
from bpmopt.bpmn import read_bpmn
from bpmopt.event_log import export_csv
from bpmopt.features import case_features
from bpmopt.lint import lint
from bpmopt.ml import cycle_time_task, ml_section, rework_tasks
from bpmopt.parameters import assumptions, load_parameters
from bpmopt.planted import load_planted_effects
from bpmopt.results import build_results, element_names, export_results
from bpmopt.scenarios import find_scenario, load_catalog, run_scenarios
from bpmopt.simulation import simulate
from bpmopt.value_added import load_value_added, time_per_class

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
ORIGINAL = ROOT / "current_process.bpmn"
MODEL = ROOT / "models/as_is.bpmn"
PARAMETERS = ROOT / "models/as_is.params.toml"
VALUE_ADDED = ROOT / "models/as_is.value_added.toml"
PLANTED = ROOT / "models/as_is.planted_effects.toml"
CATALOG = ROOT / "models/to_be.catalog.toml"
TO_BE = ROOT / "models/to_be.bpmn"  # the full to-be model, C1 and C2 (C3 and C4 only change parameters)
APPROVED = "End_ProjectClosed"  # end event of an approved project
LOG_CSV = ROOT / "notebooks/output/as_is_event_log.csv"
SEED = 42
N_CASES = 2_000  # cases kept after the warm-up, in the baseline run and in every scenario run
REPLICATIONS = 30  # runs per scenario

pd.set_option("display.precision", 2)

/Users/tonytrinh/School/IS6003/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Check the models

The linter applies the rules from §5.4 that a program can check: the structural rules of §5.4.1 (pp.183-187), the labeling rules of §5.4.3 (pp.189-190) and the seven guidelines of 7PMG in §5.4.4 (pp.192-193).

The original model, `current_process.bpmn`, is kept unchanged for reference. The corrected model, `models/as_is.bpmn`, is what we simulate. Its changes are listed in `models/as_is_changelog.md`. The original should show violations and the corrected one none.

In [2]:
original = pd.DataFrame(lint(ORIGINAL))
print(f"current_process.bpmn: {len(original)} violations")
original.groupby(["rule", "severity"]).size().rename("count").to_frame()

current_process.bpmn: 27 violations


,,count
rule,severity,
ARC-LABEL,error,8
EVENT-LABEL,error,2
G3,warning,1
G6,warning,15
POOL,error,1


In [3]:
as_is_violations = lint(MODEL)
print(f"models/as_is.bpmn: {len(as_is_violations)} violations")

models/as_is.bpmn: 0 violations


## 2. Parameters

The simulator needs an arrival rate, case attributes, task durations, branch probabilities and lane capacities. We have no real data, so every value is an estimate for a bank IT division, marked "Proposed default" in `models/as_is.params.toml`. Entries marked "Student" carry the student's answers: outright rejection of a proposal is rare (3% of decisions), and the cycle time ends at go-live. The book warns that simulation inputs should come from observation where possible and be checked with the people who run the process (§7.3.4, p.287). This is the main threat to validity, so the whole table goes on the page.

Requests arrive at 0.234 per working day, a bit under one a week (arrival rate, p.266). That rate keeps the busiest lane, QC, at about 87% busy, so the results aren't dominated by a queue close to saturation (§7.2, p.276). The first 250 working days are a warm-up and are dropped, so the queues don't start empty.

In [4]:
model = read_bpmn(MODEL)
params = load_parameters(PARAMETERS, model=model)
print(f"Parameter version: {params.version}. Cycle time ends at the end of {params.cycle_time_end}.")
pd.DataFrame(assumptions(params))

Parameter version: proposed-3. Cycle time ends at the end of Task_StartOperation.


,parameter,value,rationale
0,arrivals.rate,0.23,Proposed default. About 58 new requests a year...
1,warm_up.length,250.0,"Proposed default. One working year, long enoug..."
2,cycle_time.ends_at,Task_StartOperation,Student decision. The cycle time ends at go-li...
3,attributes.project_type.distribution,"choice(enhancement: 0.55, new_system: 0.25, re...",Proposed default. Most requests change an exis...
4,attributes.complexity.distribution,"choice(1: 0.35, 2: 0.45, 3: 0.2)","Proposed default. Ordinal scale 1 low, 2 mediu..."
5,attributes.integrations.distribution,poisson(2),Proposed default. Number of other systems the ...
6,attributes.security_level.distribution,"choice(1: 0.4, 2: 0.4, 3: 0.2)",Proposed default. Data classification: 1 inter...
7,attributes.requesting_unit.distribution,"choice(retail: 0.35, corporate: 0.2, treasury:...",Proposed default. Retail banking sends the mos...
8,attributes.team_workload.distribution,"triangular(0.5, 0.8, 1)",Proposed default. Share of the development tea...
9,lanes.Lane_BusinessUnit.capacity,5,Proposed default. Five business analysts or PM...


## 3. Simulate and write the event log

The simulator runs the BPMN file itself, so the diagram is exactly the process that is simulated. Each lane is a pool of identical resources with the capacity above, so cases queue when a lane is busy.

The log has one row per element execution, with the columns of the book's event log: case ID, activity, resource (here the lane) and time (§11.3, p.422, Fig. 11.4 on p.423). Instead of one timestamp we keep ready, start and end times, because the book needs start and end times to tell processing time from waiting time (§11.5, pp.443-444). The case attributes come last, as extra data attributes (p.422). XOR-split decisions are zero-length rows with the flow taken, so rework can be counted.

The CSV goes to `notebooks/output/`, which is not committed. The notebook rewrites it on every run.

In [5]:
log = simulate(model, params, n_cases=N_CASES, seed=SEED)
LOG_CSV.parent.mkdir(exist_ok=True)
export_csv(log, LOG_CSV)
print(f"{log.case_id.nunique():,} cases, {len(log):,} rows, written to {LOG_CSV.relative_to(ROOT)}")
log.head(8)

2,000 cases, 54,542 rows, written to notebooks/output/as_is_event_log.csv


,case_id,activity_id,activity_label,element_type,lane_id,lane_label,ready_time,start_time,end_time,flow_id,project_type,complexity,integrations,security_level,requesting_unit,team_workload
0,65,Start_RequestReceived,Development request received,startEvent,Lane_BusinessUnit,Business unit / PMO,250.69,250.69,250.69,NaN,regulatory,1,1,2,operations,0.77
1,65,Task_RegisterRequest,Register development request,task,Lane_BusinessUnit,Business unit / PMO,250.69,257.69,259.03,NaN,regulatory,1,1,2,operations,0.77
2,65,Task_ApproveProposal,Approve project proposal,task,Lane_ITManagement,IT management,259.03,259.03,263.11,NaN,regulatory,1,1,2,operations,0.77
3,65,Gateway_ProposalSplit,,exclusiveGateway,Lane_ITManagement,IT management,263.11,263.11,263.11,Flow_ProposalApproved,regulatory,1,1,2,operations,0.77
4,65,Task_HoldKickOff,Hold project kick-off,task,Lane_BusinessUnit,Business unit / PMO,263.11,263.11,264.22,NaN,regulatory,1,1,2,operations,0.77
5,65,Task_SpecifyRequirements,Specify business and technical requirements,task,Lane_BusinessUnit,Business unit / PMO,264.22,264.22,275.97,NaN,regulatory,1,1,2,operations,0.77
6,65,Task_DraftPlan,Draft detailed project plan,task,Lane_Development,Application development,275.97,278.11,283.71,NaN,regulatory,1,1,2,operations,0.77
7,65,Task_ApprovePlan,Approve project plan,task,Lane_ITManagement,IT management,283.71,284.25,285.85,NaN,regulatory,1,1,2,operations,0.77


## 4. Baseline

From the event log:

- Cycle time (p.256) runs from the request to go-live, the end of "Start live operation" (why, see below). A case that ends earlier, at "Proposal rejected", ends at that end event.
- Processing time is the time in which some task of the case is in progress, up to go-live. Parallel work counts once, as in the book's theoretical cycle time (p.262).
- Waiting time is the cycle time minus processing time, the time "where no work is being done to advance the process" (p.261).
- Cycle time efficiency is the mean processing time over mean cycle time (Eq. 7.5, p.262).

Why go-live: the business unit has its application once it is live. "Monitor live operation" is about two weeks of hypercare and "Evaluate project results" is the closing report. Both stay in the model and the log, but no redesign aims at them, so they would only dilute the saving. The cut-off is one entry in the parameter file (`[cycle_time]`), and the baseline, the flow analysis check, the ML and the scenario runner all read it from there.

The headline counts every case, rejected proposals included. Approved projects (cases that end at "Project closed") are shown next to it as a check.

In [6]:
baseline = baseline_metrics(log, model, ends_at=params.cycle_time_end)
pd.DataFrame({k: baseline[k] for k in ("cycle_time", "processing_time", "waiting_time")}).T

,mean,median,p90
cycle_time,132.51,122.88,197.96
processing_time,104.73,99.39,147.68
waiting_time,27.78,20.44,61.26


In [7]:
names = element_names(model)

print(f"Cycle time efficiency: {baseline['cycle_time_efficiency']:.2f}")
print(f"Waiting share of the mean cycle time: {baseline['waiting_time']['mean'] / baseline['cycle_time']['mean']:.0%}")
pd.DataFrame(
    {names[o["end_event_id"]]: {"cases": o["case_count"], **o["cycle_time"], "efficiency": o["cycle_time_efficiency"]} for o in baseline["by_outcome"]}
).T

Cycle time efficiency: 0.79
Waiting share of the mean cycle time: 21%


,cases,mean,median,p90,efficiency
Project closed,1924.0,137.27,124.33,200.27,0.79
Proposal rejected,76.0,12.00,9.15,22.54,0.66


Cycle time efficiency is high for a real process. The book's Table 7.4 (p.263) reports values around 5% or lower. Ours is high because the simulation only has queueing for lanes with a few identical resources. It has no handoff delays, batching, holidays or people working on several projects at once, which are where most waiting comes from in practice (p.261). Read this as a lower bound on waiting.

### Rework and waiting per activity

Rework rate per rework gateway: the share of the cases that reached the gateway and looped back at least once. Rework loops are defect waste (§6.2, p.222) and lengthen the cycle time by a factor of 1 / (1 - r) (Eq. 7.4, p.260). Gateways in the model are unnamed, so `element_names` names each one after the task just before it, the same names the page shows.

In [8]:
rework = pd.DataFrame(baseline["rework"])
rework.insert(1, "decision", rework.gateway_id.map(names))
rework

,gateway_id,decision,rate
0,Gateway_ReleaseSplit,Decision after Approve application release,0.08
1,Gateway_TestSplit,Decision after Run acceptance tests,0.37
2,Gateway_PlanSplit,Decision after Approve project plan,0.19
3,Gateway_ProposalSplit,Decision after Approve project proposal,0.20


Mean waiting per activity run, from ready to start, longest first. This is where cases queue for a free resource (waiting waste, §6.2, pp.221-222).

In [9]:
waiting = pd.DataFrame(baseline["waiting_by_activity"])
waiting.insert(1, "activity", waiting.activity_id.map(names))
waiting

,activity_id,activity,mean
0,Task_RunAcceptanceTests,Run acceptance tests,1.97
1,Task_RunSecurityTests,Run security tests,1.96
2,Task_RunFunctionalTests,Run functional tests,1.94
3,Task_ReviewSourceCode,Review source code,1.89
4,Task_ApprovePlan,Approve project plan,1.61
5,Task_ApproveProposal,Approve project proposal,1.61
6,Task_HoldKickOff,Hold project kick-off,1.58
7,Task_RegisterRequest,Register development request,1.57
8,Task_ApproveRelease,Approve application release,1.49
9,Task_RefineApplication,Refine application,1.42


## 5. Flow analysis check

The book computes the theoretical cycle time from the model and the mean task durations with four block formulas: sequence, XOR, AND and rework (§7.1.1, pp.256-260). Flow analysis ignores resource contention (§7.1.7, p.273), so we compare it with a second simulation that has unlimited capacity in every lane, with the same seed and case count. Both stop at go-live, and a rejected case at its end event, so they cover the same span. The two should agree within sampling error. Branch probabilities that depend on case attributes are averaged over attribute draws, which is our extension of the book's formulas.

In [10]:
check = flow_analysis_check(model, params, n_cases=N_CASES, seed=SEED)
print(f"Book formula: {check['theoretical']:.1f} days. Simulated with unlimited capacity: {check['simulated']:.1f} days. "
      f"Difference: {check['simulated'] / check['theoretical'] - 1:+.1%}")

Book formula: 105.4 days. Simulated with unlimited capacity: 104.7 days. Difference: -0.6%


## 6. Value-added classification

Each task gets the class of its main step: value-adding (VA), business value-adding (BVA) or non-value-adding (NVA), as defined on pp.214-215. The classes are a draft awaiting the student's review (`docs/analysis/phase1-qualitative.md`). The table below adds how much task time each class takes on the simulated log, per case, up to go-live.

In [11]:
value_added = load_value_added(VALUE_ADDED, model)
time_per_class(log, value_added, ends_at=params.cycle_time_end)

,tasks,days_per_case,share
class,,,
VA,4,52.99,0.51
BVA,12,47.11,0.45
NVA,3,4.63,0.04


## 7. Machine learning

The book has no chapter on predictive ML. The method is ours and is set out in the spec:

- Features are the case attributes drawn at arrival, the only things known when a request comes in. One row per case.
- The split follows case start time. The last 20% of cases are the test set. Training uses the first 80%, minus any case that was still open when the first test case started, so no model learns from later cases.
- The models are a reference that ignores the inputs (mean, or majority class), a linear or logistic regression, and gradient boosting.
- Metrics are MAE and R² for cycle time, ROC-AUC and F1 for rework.
- Drivers are ranked by the mean absolute SHAP value of the gradient boosting model on the test cases, one value per attribute.

Task A predicts the cycle time, up to go-live, and only for approved projects: it trains and tests on cases that end at "Project closed". A rejected proposal ends early for a reason no attribute explains (the split is fixed), so mixing those cases in would mostly add noise. Task B has one classifier per rework gateway and predicts whether a case loops back there at least once. Only cases that reached the gateway count.

In [12]:
case_features(log).head()

,start_time,project_type,complexity,integrations,security_level,requesting_unit,team_workload
case_id,,,,,,,
65,250.69,regulatory,1,1,2,operations,0.77
66,259.69,enhancement,1,2,2,risk_compliance,0.82
67,266.72,new_system,2,4,2,treasury,0.65
68,271.18,regulatory,2,4,1,operations,0.76
69,273.71,enhancement,2,3,2,treasury,0.60


In [13]:
tasks = [
    cycle_time_task(log, seed=SEED, ends_at=params.cycle_time_end, outcome=APPROVED),
    *rework_tasks(log, model, seed=SEED),
]


def title(task):
    return "cycle time" if task["kind"] == "regression" else f"rework, {names[task['gateway_id']]}"


fitted = [t for t in tasks if "skipped" not in t]
pd.DataFrame({m["id"]: m["metrics"] for t in fitted if t["kind"] == "regression" for m in t["models"]}).T

,mae,r2
reference,31.80,-2.00e-03
linear,30.10,1.08e-01
gradient_boosting,30.22,1.01e-01


In [14]:
pd.DataFrame({(title(t), m["id"]): m["metrics"] for t in fitted if t["kind"] == "classification" for m in t["models"]}).T

roc_auc  \
rework, Decision after Approve application release reference             0.50   
                                                   linear                0.69   
                                                   gradient_boosting     0.52   
rework, Decision after Run acceptance tests        reference             0.50   
                                                   linear                0.63   
                                                   gradient_boosting     0.64   
rework, Decision after Approve project plan        reference             0.50   
                                                   linear                0.55   
                                                   gradient_boosting     0.56   
rework, Decision after Approve project proposal    reference             0.50   
                                                   linear                0.47   
                                                   gradient_boosting     0.51   

                                                                        f1  
rework, Decision after Approve application release reference          0.00  
                                                   linear             0.24  
                                                   gradient_boosting  0.17  
rework, Decision after Run acceptance tests        reference          0.00  
                                                   linear             0.53  
                                                   gradient_boosting  0.53  
rework, Decision after Approve project plan        reference          0.00  
                                                   linear             0.32  
                                                   gradient_boosting  0.32  
rework, Decision after Approve project proposal    reference          0.00  
                                                   linear             0.29  
                                                   gradient_boosting  0.29

A rework gateway where the training or the test cases all went the same way can't be fitted or scored. The package then skips that gateway and records why, instead of stopping the run. The page shows the reason in place of the scores.

In [15]:
skipped = {t["id"]: t["skipped"] for t in tasks if "skipped" in t}
print(skipped or "No gateway skipped with this seed.")

No gateway skipped with this seed.


Drivers per task, ranked by mean |SHAP|, rank 1 = strongest. For cycle time the values are in days, for rework in log-odds.

In [16]:
drivers = pd.DataFrame({title(t): {d["feature"]: d["mean_abs_shap"] for d in t["drivers"]} for t in fitted})
ranks = drivers.rank(ascending=False).astype(int)
drivers.round(2).astype(str) + "  (#" + ranks.astype(str) + ")"

,cycle time,"rework, Decision after Approve application release","rework, Decision after Run acceptance tests","rework, Decision after Approve project plan","rework, Decision after Approve project proposal"
complexity,11.04 (#1),0.11 (#3),0.37 (#1),0.07 (#4),0.01 (#6)
security_level,7.37 (#2),0.01 (#6),0.25 (#2),0.03 (#6),0.09 (#3)
integrations,4.94 (#3),0.06 (#5),0.22 (#3),0.06 (#5),0.1 (#2)
team_workload,3.01 (#4),0.33 (#1),0.1 (#4),0.2 (#2),0.2 (#1)
project_type,1.38 (#5),0.23 (#2),0.06 (#5),0.29 (#1),0.03 (#5)
requesting_unit,0.76 (#6),0.1 (#4),0.03 (#6),0.07 (#3),0.06 (#4)


## 8. ML drivers against the planted effects

The log is synthetic, so we know the truth. `models/as_is.planted_effects.toml` lists every attribute effect put into the parameters, and `models/as_is_planted_effects.md` explains each one. Loading the file checks it against the parameters, and the package works out from it which attributes each ML task should find. `requesting_unit` affects nothing and is the negative control: if the ML ranks it high, the ranking is picking up noise. The table puts each task's planted attributes next to the ranks the ML gave them.

In [17]:
planted = load_planted_effects(PLANTED, params)
expected = {title(t): planted.expected_drivers(model)[t["id"]] for t in fitted}
control = planted.negative_control

pd.DataFrame(
    {
        "planted": {task: ", ".join(attrs) or "(nothing)" for task, attrs in expected.items()},
        "their ranks": {task: ", ".join(f"#{ranks.loc[a, task]}" for a in attrs) or "-" for task, attrs in expected.items()},
        "top 3 found": {task: ", ".join(ranks[task].sort_values().index[:3]) for task in expected},
        f"{control} rank": {task: f"#{ranks.loc[control, task]} of {len(ranks)}" for task in expected},
    }
)

,planted,their ranks,top 3 found,requesting_unit rank
cycle time,"complexity, integrations, team_workload, secur...","#1, #3, #4, #2, #5","complexity, security_level, integrations",#6 of 6
"rework, Decision after Approve application release",project_type,#2,"team_workload, project_type, complexity",#4 of 6
"rework, Decision after Run acceptance tests","complexity, integrations, security_level","#1, #3, #2","complexity, security_level, integrations",#6 of 6
"rework, Decision after Approve project plan",project_type,#1,"project_type, team_workload, requesting_unit",#3 of 6
"rework, Decision after Approve project proposal",(nothing),-,"team_workload, integrations, security_level",#4 of 6


What this shows with seed 42 and 2,000 cases. The cycle time task uses the 1,924 approved projects: 385 test cases, and 1,522 training cases, since 17 of the first 1,539 were still open when the test cases started. The rework tasks use every case that reached the gateway.

- For cycle time, the drivers come out in the planted order of strength: complexity, security level, integrations, then team workload and project type. The negative control, `requesting_unit`, ranks last.
- At the test loop the top three drivers are exactly the three planted ones: complexity, security level and integrations. At the plan loop, project type comes first, which is the one planted effect there.
- At the release loop project type comes second, behind team workload, which has no effect there. Only about 8% of releases are rejected, so the test set has few positive cases.
- The negative control never ranks first. It is last for cycle time and at the test loop, third at the plan loop and fourth at the release and proposal loops. At the proposal loop nothing is planted, all attributes get small values and the order among them means little.
- The cycle time models beat predicting the mean, but only just: R² is about 0.1 for both on the test cases (the reference scores 0), and the MAE drops from 31.8 to about 30.1 days. Most of a project's cycle time comes from chance: how many times work loops back and how long the queues are. No attribute known at the start explains that. Even a linear fit scored on the same approved projects it was fitted on explains only about 15% of the variation (`models/as_is_planted_effects.md`).
- Keeping the rejected proposals out helps. They end early for a reason no attribute explains, since the proposal decision has fixed probabilities. In the first run (all cases, monitoring counted, 10% rejected) R² was about 0.
- The rework models clearly beat the reference only at the test loop (ROC-AUC about 0.63) and, for the logistic model, at the release loop (0.69). The plan loop scores about 0.55, and the proposal loop, where nothing is planted, about 0.5.
- Team workload is the only continuous attribute. Trees can split it in many places, so it picks up some SHAP value even where it has no effect. Small SHAP values for it shouldn't be read as a driver.

This is a threat to validity. It's a check of the method, not a finding about the real process. The ML can only find effects we put in, and it finds the strong ones. The weak effects (project type and team workload on cycle time) rank above the negative control here, but not by much, and with 2,000 cases another seed could swap them. On real data, a driver that ranks no higher than an attribute known to have no effect shouldn't be used to justify a redesign. For the redesign, the evidence that holds is complexity, security level and integrations in the cycle time and at the test loop, and project type at the plan loop.

## 9. Redesign

The redesign is a human decision (Phase 5). The student delegated the design to Claude and approved all four changes on 2026-11-03. Each change targets a finding above and applies a heuristic from Appendix A. Its trade-off is stated on the Devil's Quadrangle (§8.1.3, p.303). Why each change exists and what it costs is written up in `models/to_be_changelog.md`.

- C1 runs the four QC checks in parallel (Parallelism, A.3). QC is the main queue.
- C2 retests only what was fixed instead of running all four checks again (Activity elimination, A.2). The test loop has the highest rework rate.
- C3 gives complex and highly classified cases a longer requirements step, with fewer defects later (Triage, A.2). Complexity and security level are the two strongest ML drivers.
- C4 adds one QC tester (Extra resources, A.4).

C1 and C2 change the model, so each combination of them has its own BPMN file. C3 and C4 only change parameters. The catalog file says which model and which parameter overrides each change brings. Loading it resolves all 16 combinations and checks each one against its model, so a broken one fails here, before any simulation.

In [18]:
catalog = load_catalog(CATALOG, model=model, params=params)
pd.DataFrame(catalog.entries()).set_index("id")[["title", "driver", "heuristic", "trade_off"]]

,title,driver,heuristic,trade_off
id,,,,
C1,Run the four checks in parallel,"Baseline run (seed 42, 2,000 cases): the test ...","{'name': 'Parallelism', 'ref': 'Appendix A.3'}","{'time': 'better', 'cost': 'worse', 'quality':..."
C2,Retest only what was fixed,"Baseline run (seed 42, 2,000 cases): Gateway_T...","{'name': 'Activity elimination', 'ref': 'Appen...","{'time': 'better', 'cost': 'better', 'quality'..."
C3,Triage risky cases into an extended requiremen...,"ML on the baseline run (ticket 14, seed 42, 2,...","{'name': 'Triage', 'ref': 'Appendix A.2'}","{'time': 'better', 'cost': 'worse', 'quality':..."
C4,Add one QC tester,"Baseline run (seed 42, 2,000 cases): QC is the...","{'name': 'Extra resources', 'ref': 'Appendix A...","{'time': 'better', 'cost': 'worse', 'quality':..."


The to-be models get the same rule check as the as-is model. They have zero errors. Each has one warning: they are just over the 30 elements of 7PMG G7 (§5.4.4, p.193). The cell below prints how many flow nodes each has. The book's fix is a sub-process, which our simulator doesn't support, so the student accepted the warning. It goes into the threats to validity.

In [19]:
for file in sorted((ROOT / "models").glob("to_be*.bpmn")):
    violations = lint(file)
    print(f"{file.name}: {len(read_bpmn(file).nodes)} flow nodes, " + (", ".join(f"{v.rule} ({v.severity})" for v in violations) or "clean"))

to_be.bpmn: 33 flow nodes, G7 (warning)
to_be_c1.bpmn: 32 flow nodes, G7 (warning)
to_be_c2.bpmn: 31 flow nodes, G7 (warning)


## 10. Evaluate the redesign

The scenario runner simulates the as-is process and all 2^4 = 16 combinations of changes. The set-up is the same as the baseline run: the same parameter file, the same 2,000 cases per run after the same 250-day warm-up, and seeds derived from the same seed (42). Each scenario runs 30 times. Run r of every scenario uses the same seed, so every scenario sees the same arrivals, case attributes and draws per task and gateway. This is the method of common random numbers: differences between scenarios come from the design, not from luck. The book doesn't cover warm-up periods, replications or common random numbers; they are standard practice in discrete-event simulation, and the spec asks for them.

For each scenario we report the mean, median and 90th percentile cycle time, averaged over the 30 runs, with 95% t-intervals. The saving is the as-is mean minus the scenario's mean, paired by run, so its interval only reflects how the difference varies. The share faster is the mean saving divided by the as-is mean cycle time, and its interval is the saving's interval divided by the same mean, so a share always matches the days next to it.

`build_results` then assembles the results file: run settings, page text, assumptions (the as-is parameters, then each change's overrides, marked with the change), threats, baseline, value-added classes, ML, change catalog, scenarios, and the ablation and headline derived from them. The rest of this section reads from it.

This cell takes about 6 minutes.

In [20]:
scenarios = run_scenarios(catalog, n_cases=N_CASES, seed=SEED, replications=REPLICATIONS)
results = build_results(
    log=log, model=model, params=params, seed=SEED, value_added=value_added, ml=ml_section(tasks), catalog=catalog, scenarios=scenarios
)


def label(changes):
    return "+".join(changes) or "as-is"


pd.DataFrame(
    {
        label(s["changes"]): {
            "mean": s["cycle_time"]["mean"]["value"],
            "median": s["cycle_time"]["median"]["value"],
            "P90": s["cycle_time"]["p90"]["value"],
            "saving (days)": s["saving"]["value"],
            "saving CI": f"[{s['saving']['ci_low']:.1f}, {s['saving']['ci_high']:.1f}]",
            "faster": f"{s['saving_share']['value']:.1%}",
            "faster CI": f"[{s['saving_share']['ci_low']:.1%}, {s['saving_share']['ci_high']:.1%}]",
        }
        for s in sorted(scenarios, key=lambda s: -s["saving"]["value"])
    }
).T

,mean,median,P90,saving (days),saving CI,faster,faster CI
C1+C2+C4,101.71,99.39,138.75,39.09,"[34.8, 43.4]",27.8%,"[24.7%, 30.8%]"
C1+C2,102.41,99.9,140.35,38.39,"[34.1, 42.7]",27.3%,"[24.2%, 30.3%]"
C1+C2+C3+C4,104.55,102.93,142.46,36.25,"[32.0, 40.5]",25.7%,"[22.7%, 28.8%]"
C1+C2+C3,105.05,103.32,143.35,35.75,"[31.5, 40.0]",25.4%,"[22.4%, 28.4%]"
C1+C3+C4,107.63,105.39,148.05,33.17,"[29.0, 37.3]",23.6%,"[20.6%, 26.5%]"
C1+C4,109.24,104.35,154.12,31.56,"[27.7, 35.4]",22.4%,"[19.7%, 25.1%]"
C1+C3,109.4,106.81,151.25,31.4,"[27.4, 35.4]",22.3%,"[19.5%, 25.1%]"
C2+C4,113.03,110.76,151.88,27.77,"[23.5, 32.1]",19.7%,"[16.7%, 22.8%]"
C2,113.28,111.01,152.29,27.52,"[23.3, 31.8]",19.5%,"[16.5%, 22.6%]"
C2+C3+C4,115.84,114.28,155.25,24.96,"[20.7, 29.2]",17.7%,"[14.7%, 20.7%]"


### The baseline is one of many possible runs

The baseline in section 4 is one run with seed 42. The as-is scenario here averages 30 runs of the same process with the same settings, so the two means differ. The gap is run-to-run variation: QC runs close to its limit (the arrival rate is set for that, section 2), and a run that happens to get a few complex cases close together builds a long queue that lifts its whole mean (§7.2.2, p.276, on why waiting grows fast as utilisation nears 100%). The baseline stays the right source for *where* time goes. The evaluation is the right source for *how much* a change saves.

In [21]:
as_is_runs = pd.Series(find_scenario(scenarios, [])["replication_means"])
print(f"Baseline run (seed {SEED}): {baseline['cycle_time']['mean']:.1f} days")
print(f"As-is over {REPLICATIONS} runs: {as_is_runs.mean():.1f} days, runs from {as_is_runs.min():.1f} to {as_is_runs.max():.1f}, "
      f"the baseline run is lower than {(as_is_runs > baseline['cycle_time']['mean']).mean():.0%} of them")
as_is_runs.describe().to_frame("as-is mean per run").T

Baseline run (seed 42): 132.5 days
As-is over 30 runs: 140.8 days, runs from 124.0 to 200.5, the baseline run is lower than 77% of them


,count,mean,std,min,25%,50%,75%,max
as-is mean per run,30.0,140.8,14.61,123.96,132.67,137.11,142.62,200.52


### Ablation

The ablation reads two views off the scenarios:

- `alone`: the saving of the scenario with only that change;
- `full_minus`: how many days longer the full design gets when that change is dropped. A negative value means the full design is faster without it. Its interval pairs the two scenarios run by run.

When changes don't overlap, the two views agree. When they target the same queue, a change saves less on top of the others than it does alone.

In [22]:
pd.DataFrame(
    {
        row["change"]: {
            "alone": row["alone"]["value"],
            "alone CI": f"[{row['alone']['ci_low']:.1f}, {row['alone']['ci_high']:.1f}]",
            "full minus": row["full_minus"]["value"],
            "full minus CI": f"[{row['full_minus']['ci_low']:.1f}, {row['full_minus']['ci_high']:.1f}]",
        }
        for row in results["ablation"]
    }
).T

,alone,alone CI,full minus,full minus CI
C1,20.36,"[19.4, 21.4]",11.29,"[11.1, 11.5]"
C2,27.52,"[23.3, 31.8]",3.08,"[2.9, 3.3]"
C3,15.61,"[11.7, 19.5]",-2.84,"[-3.2, -2.4]"
C4,13.0,"[9.4, 16.6]",0.5,"[0.3, 0.7]"


In [23]:
summary = results["headline"]
rows = {row["change"]: row for row in results["ablation"]}
as_is, full, best = (find_scenario(scenarios, changes) for changes in ([], summary["changes"], summary["best"]))


def ci(estimate, spec=".1f"):
    return f"{estimate['ci_low']:{spec}} to {estimate['ci_high']:{spec}}"


share, top = full["saving_share"], rows[summary["top_change"]]["full_minus"]
print(f"Full design {label(summary['changes'])}: {share['value']:.1%} faster (95% CI {ci(share, '.1%')}). "
      f"Mean cycle time {as_is['cycle_time']['mean']['value']:.1f} -> {full['cycle_time']['mean']['value']:.1f} days, "
      f"a saving of {full['saving']['value']:.1f} days (CI {ci(full['saving'])}).")
print(f"Contributes most to the full design: {summary['top_change']}, {top['value']:.1f} days added if dropped (CI {ci(top)}).")
print(f"Best combination: {label(summary['best'])}, {best['saving_share']['value']:.1%} faster (CI {ci(best['saving_share'], '.1%')}).")
print(f"The savings alone add up to {sum(row['alone']['value'] for row in rows.values()):.1f} days, "
      f"against {full['saving']['value']:.1f} for all changes together.")
print(f"As-is mean cycle time CI: {ci(as_is['cycle_time']['mean'])} days. Saving CI of each change alone: "
      + ", ".join(f"{c} {ci(row['alone'])}" for c, row in rows.items()) + ".")

Full design C1+C2+C3+C4: 25.7% faster (95% CI 22.7% to 28.8%). Mean cycle time 140.8 -> 104.5 days, a saving of 36.3 days (CI 32.0 to 40.5).
Contributes most to the full design: C1, 11.3 days added if dropped (CI 11.1 to 11.5).
Best combination: C1+C2+C4, 27.8% faster (CI 24.7% to 30.8%).
The savings alone add up to 76.5 days, against 36.3 for all changes together.
As-is mean cycle time CI: 135.3 to 146.3 days. Saving CI of each change alone: C1 19.4 to 21.4, C2 23.3 to 31.8, C3 11.7 to 19.5, C4 9.4 to 16.6.


### What the evaluation shows

Read with the numbers above (seed 42, 2,000 cases per run, 30 runs per scenario):

- The full design meets the objective. The whole interval of its saving share is above the 20% target from Phase 1.
- Which change contributes most depends on how you ask. In the full design it is C1: dropping it adds the most days. On its own, C2 saves the most (see the ablation table). The two views disagree because C1 and C2 work on the same QC queue. Once one of them has shortened it, the other has less left to remove. The headline names C1, because the question is what the proposed design needs.
- C3 hurts once C1 and C2 are on. It saves time on its own, but its full-minus interval is below zero, so the full design is faster without it. With the QC queue drained, fewer defects save little, while the longer requirements step still loads the business unit lane (ticket 15's utilisation check, 1,000 cases x 10 runs, saw both effects).
- C4 adds little on top. One more tester saves a lot alone but close to nothing in the full design, for the cost of a salary.
- The best combination isn't the full design. We still report the full design as the headline, because it is the design the student approved. Reporting the best of all combinations, picked after looking at them, would overstate the gain.
- The changes only partly add up. The savings alone sum to far more than the saving of all four together. Three of the four target the same queue, so the saving of each depends on what else changed. That is why the ablation looks at the full design, not only at each change alone.
- Savings are precise, cycle times less so. The as-is mean has a wide interval, each saving a narrow one. Common random numbers make the noise mostly cancel out in a paired difference.

Dropping C3 is a real option for the student: it would trade the fewer defects C3 aims for against the days in its full-minus row. The simulation can't settle that, because it counts a defect only by the rework time it causes.


## 11. Write the results for the page

`export_results` copies both models next to the results, lints them, checks everything against the JSON schema and writes `web/results.json`. The schema check includes the rules a schema can't express: the scenarios are the full factorial of the catalog, the ablation and the headline match the scenarios, and every saving share is its saving over the as-is mean.

The threats to validity are page text, in `src/bpmopt/content/page.en.toml`: the data is synthetic, the parameters and the effects of the changes are estimates, there is no working calendar, resources are identical, QC runs close to its limit, and the to-be models break G7. The page text and threats carry no typed-in results: the page fills their numbers from `web/results.json`.

In [24]:
path = export_results(results, ROOT / "web", models={"as_is": MODEL, "to_be": TO_BE})
print(f"Wrote {path.relative_to(ROOT)}")

Wrote web/results.json
